# Vectorized QMC with Bayesian Stopping Criteria

Demonstrates Bayesian cubature rules (`CubQMCBayesLatticeG` and `CubQMCBayesNetG`)
for integration, which model the integrand as a Gaussian process and provide
posterior credible intervals as error estimates.

Based on the Python QMCPy demo `vectorized_qmc_bayes.ipynb`.

In [ ]:
using QMC
import QMC: Uniform
using Statistics
using Printf

## LD Sequences Overview

Compare IID, digital net, and lattice point sets.

In [ ]:
n = 2^6
for (dd, name) in [
    (IIDStdUniform(2; seed=7),  "IID"),
    (DigitalNetB2(2; seed=7),   "Digital Net"),
    (Lattice(2; seed=7),        "Lattice"),
]
    pts = gen_samples(dd, n)
    println("$name ($n points):")
    println("  Mean:  $(round.(mean(pts, dims=1), digits=3))")
    println("  Range: [$(round.(minimum(pts, dims=1), digits=3)), " *
            "$(round.(maximum(pts, dims=1), digits=3))]")
end

## Cantilever Beam Example

3D input → 2D output: displacement D and stress S as functions of
Young's modulus E, horizontal load X, and vertical load Y.

In [ ]:
function cantilever_beam(x)
    l, w, t = 100.0, 4.0, 2.0
    E, X, Y = x[1], x[2], x[3]
    D = 4l^3 / (E * w * t) * sqrt(X^2/t^4 + Y^2/w^4)
    S = 6l / (w * t^2) * sqrt(X^2 + Y^2)
    return D, S
end

# Evaluate over points
dd = DigitalNetB2(3; seed=7)
x = gen_samples(dd, 1024)
D_vals = Float64[]
S_vals = Float64[]
for i in 1:size(x, 1)
    disp, stress = cantilever_beam(x[i, :])
    push!(D_vals, disp)
    push!(S_vals, stress)
end

println("Displacement D: mean=$(round(mean(D_vals), digits=4)), std=$(round(std(D_vals), digits=4))")
println("Stress S:       mean=$(round(mean(S_vals), digits=4)), std=$(round(std(S_vals), digits=4))")

## Bayesian QMC Integration

Use the Bayesian stopping criteria which model the integrand as a sample path
from a Gaussian process to obtain credible-interval-based error bounds.

In [ ]:
# --- Bayesian Lattice ---
println("=== CubQMCBayesLatticeG ===")
let
    for dim in [1, 3, 5]
        dd_lattice = Lattice(dim; seed=7)
        tm_lattice = Gaussian(dd_lattice; mean=0.0, covariance=0.5)
        f_lattice = Keister(tm_lattice)
        sc_lattice = CubQMCBayesLatticeG(f_lattice; abs_tol=1e-3)
        result_lattice = integrate(sc_lattice)
        exact_lattice = keister_exact(dim)
        err = abs(result_lattice.solution - exact_lattice)
        println("  d=$dim: sol=$(round(result_lattice.solution, digits=6)), " *
                "err=$(round(err, sigdigits=2)), " *
                "n=$(result_lattice.data[:n])")
    end
end

In [ ]:
# --- Bayesian Digital Net ---
println("=== CubQMCBayesNetG ===")
let
    for dim in [1, 3, 5]
        dd_net = DigitalNetB2(dim; seed=7)
        tm_net = Gaussian(dd_net; mean=0.0, covariance=0.5)
        f_net = Keister(tm_net)
        sc_net = CubQMCBayesNetG(f_net; abs_tol=1e-3)
        result_net = integrate(sc_net)
        exact_net = keister_exact(dim)
        err = abs(result_net.solution - exact_net)
        println("  d=$dim: sol=$(round(result_net.solution, digits=6)), " *
                "err=$(round(err, sigdigits=2)), " *
                "n=$(result_net.data[:n])")
    end
end

## Comparison: Bayesian vs Frequentist QMC

Compare the Bayesian stopping criteria with their frequentist counterparts
(`CubQMCLatticeG`, `CubQMCNetG`) for the same tolerance.

In [ ]:
d = 3
tol = 1e-3
exact = keister_exact(d)

println("Keister integral (d=$d), abs_tol=$tol\n")
println("Method                      Solution     Error       n")
println("-"^60)

let
    for (name, make_sc) in [
        ("CubQMCLatticeG",      () -> begin
            dd_cmp = Lattice(d; seed=7)
            tm_cmp = Gaussian(dd_cmp; mean=0.0, covariance=0.5)
            CubQMCLatticeG(Keister(tm_cmp); abs_tol=tol)
        end),
        ("CubQMCBayesLatticeG", () -> begin
            dd_cmp = Lattice(d; seed=7)
            tm_cmp = Gaussian(dd_cmp; mean=0.0, covariance=0.5)
            CubQMCBayesLatticeG(Keister(tm_cmp); abs_tol=tol)
        end),
        ("CubQMCNetG",          () -> begin
            dd_cmp = DigitalNetB2(d; seed=7, randomize="LMS_DS", graycode=false)
            tm_cmp = Gaussian(dd_cmp; mean=0.0, covariance=0.5)
            CubQMCNetG(Keister(tm_cmp); abs_tol=tol)
        end),
        ("CubQMCBayesNetG",     () -> begin
            dd_cmp = DigitalNetB2(d; seed=7)
            tm_cmp = Gaussian(dd_cmp; mean=0.0, covariance=0.5)
            CubQMCBayesNetG(Keister(tm_cmp); abs_tol=tol)
        end),
    ]
        sc_cmp = make_sc()
        result_cmp = integrate(sc_cmp)
        err = abs(result_cmp.solution - exact)
        @printf("%-28s %.6f    %.2e    %d\n", name, result_cmp.solution, err, result_cmp.data[:n])
    end
end